<a href="https://colab.research.google.com/github/amymaaaaaa/busn32120-hw/blob/main/SQL_Queries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BUS 32120, Final Project | SQL Portion

## ***What Drives Box Office Success? A Data-Driven Analysis of the Film Features That Shape Revenue and Profitability***

### By Vivian Yang and Amy Ma

## **Introduction**

These SQL queries analyze the relationship between Oscar success and commercial success, and tries to elucidate the genres and talent which producers can optimize for success in both ends.

Note that there is a significant portion of data cleaning at the beginning. SQL starts about 30% through the notebook

In [ ]:
import requests
import pandas as pd
import numpy as np
import time
import re
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

In [ ]:
movie_df = pd.read_csv('TMDB  IMDB Movies Dataset.csv', engine='python', on_bad_lines='skip')
movie_df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2793340,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2491428,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3141794,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ...",James Cameron,James Cameron,7.9,1488308,"Sam Worthington, Zoe Saldaña, Sigourney Weaver..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com...",Joss Whedon,"Joss Whedon, Zak Penn",8.0,1549290,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ..."


In [ ]:
oscar_df = pd.read_csv('/content/Oscar Award Nominees Dataset.csv')
oscar_df.head()

,year_film,year_ceremony,ceremony,category,canon_category,name,film,winner
0,1927,1928,1,ACTOR,ACTOR IN A LEADING ROLE,Richard Barthelmess,The Noose,False
1,1927,1928,1,ACTOR,ACTOR IN A LEADING ROLE,Richard Barthelmess,The Patent Leather Kid,False
2,1927,1928,1,ACTOR,ACTOR IN A LEADING ROLE,Emil Jannings,The Last Command,True
3,1927,1928,1,ACTOR,ACTOR IN A LEADING ROLE,Emil Jannings,The Way of All Flesh,True
4,1927,1928,1,ACTRESS,ACTRESS IN A LEADING ROLE,Louise Dresser,A Ship Comes In,False


In [ ]:
movie_df.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage',
       'tconst', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'keywords', 'directors', 'writers', 'averageRating', 'numVotes',
       'cast'],
      dtype='object')

In [ ]:
oscar_df.columns

Index(['year_film', 'year_ceremony', 'ceremony', 'category', 'canon_category',
       'name', 'film', 'winner'],
      dtype='object')

# Data Cleaning for movie_df


## Step 1: Filter and Remove Outliers



Drop Rows with Incomplete Data

In [ ]:
movie_df.drop(movie_df.index[movie_df["runtime"] == 0], inplace=True)

In [ ]:
movie_df.drop(movie_df.index[movie_df["budget"] == 0], inplace=True)

In [ ]:
movie_df.drop(movie_df.index[movie_df["revenue"] == 0], inplace=True)

Drop Rows where Release Date is in the Past 3 Months (premature revenue data)

In [ ]:
movie_df['release_date'] = pd.to_datetime(movie_df['release_date'], errors='coerce')
movie_df.drop(movie_df.index[movie_df['release_date'] >= pd.Timestamp('2025-11-08')], inplace=True)

Filter for movies released after 2000

In [ ]:
movie_df['release_date'] = pd.to_datetime(movie_df['release_date'], errors='coerce')
movie_df = movie_df[(movie_df['release_date'].dt.year > 1980)]

In [ ]:
oscar_df['year_film'] = pd.to_datetime(oscar_df['year_film'], format='%Y')
oscar_df = oscar_df[(oscar_df['year_film'].dt.year > 1980)]

## Step 2: Remove Missing Values

In [ ]:
missing = movie_df.isnull().sum()
missing.name = "Missing Count"
missing

,Missing Count
id,0
title,0
vote_average,0
vote_count,0
status,0
release_date,0
revenue,0
runtime,0
adult,0
backdrop_path,118


In [ ]:
drop_cols = ['backdrop_path', 'poster_path', 'overview', 'homepage', 'tagline']
movie_df = movie_df.drop(columns=drop_cols)

In [ ]:
missing_value_cols = ['genres', 'production_companies', 'production_countries', 'spoken_languages', 'keywords', 'directors', 'writers', 'cast']
movie_df[missing_value_cols] = movie_df[missing_value_cols].fillna("Unknown")

In [ ]:
missing = movie_df.isnull().sum()
missing.name = "Missing Count"
missing

,Missing Count
id,0
title,0
vote_average,0
vote_count,0
status,0
release_date,0
revenue,0
runtime,0
adult,0
budget,0


## Step 3: Normalize titles for both datasets

In [ ]:
movie_df["title"] = (
    movie_df["title"]
    .str.lower()
    .str.strip()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
movie_df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,budget,...,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast
0,27205,inception,8.364,34495,Released,2010-07-15,825532764,148,False,160000000,...,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2793340,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,157336,interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,165000000,...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2491428,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,155,the dark knight,8.512,30619,Released,2008-07-16,1004558444,152,False,185000000,...,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3141794,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."
3,19995,avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,237000000,...,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ...",James Cameron,James Cameron,7.9,1488308,"Sam Worthington, Zoe Saldaña, Sigourney Weaver..."
4,24428,the avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,220000000,...,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com...",Joss Whedon,"Joss Whedon, Zak Penn",8.0,1549290,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ..."


In [ ]:
oscar_df["film"] = (
    oscar_df["film"]
    .str.lower()
    .str.strip()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
oscar_df.head()

,year_film,year_ceremony,ceremony,category,canon_category,name,film,winner
8183,2001-01-01,2002,74,ACTOR IN A LEADING ROLE,ACTOR IN A LEADING ROLE,Russell Crowe,a beautiful mind,False
8184,2001-01-01,2002,74,ACTOR IN A LEADING ROLE,ACTOR IN A LEADING ROLE,Sean Penn,i am sam,False
8185,2001-01-01,2002,74,ACTOR IN A LEADING ROLE,ACTOR IN A LEADING ROLE,Will Smith,ali,False
8186,2001-01-01,2002,74,ACTOR IN A LEADING ROLE,ACTOR IN A LEADING ROLE,Denzel Washington,training day,True
8187,2001-01-01,2002,74,ACTOR IN A LEADING ROLE,ACTOR IN A LEADING ROLE,Tom Wilkinson,in the bedroom,False


In [ ]:
movie_df['release_year'] = movie_df['release_date'].dt.year
movie_df = movie_df.drop(columns=['release_date'])
movie_df.head()

,id,title,vote_average,vote_count,status,revenue,runtime,adult,budget,tconst,...,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast,release_year
0,27205,inception,8.364,34495,Released,825532764,148,False,160000000,tt1375666,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2793340,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...",2010
1,157336,interstellar,8.417,32571,Released,701729206,169,False,165000000,tt0816692,...,"Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2491428,"Matthew McConaughey, Anne Hathaway, Michael Ca...",2014
2,155,the dark knight,8.512,30619,Released,1004558444,152,False,185000000,tt0468569,...,"DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3141794,"Christian Bale, Heath Ledger, Aaron Eckhart, M...",2008
3,19995,avatar,7.573,29815,Released,2923706026,162,False,237000000,tt0499549,...,"Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ...",James Cameron,James Cameron,7.9,1488308,"Sam Worthington, Zoe Saldaña, Sigourney Weaver...",2009
4,24428,the avengers,7.710,29166,Released,1518815515,143,False,220000000,tt0848228,...,Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com...",Joss Whedon,"Joss Whedon, Zak Penn",8.0,1549290,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ...",2012


In [ ]:
oscar_df['release_year'] = oscar_df['year_film'].dt.year
oscar_df = oscar_df.drop(columns=['year_film'])
oscar_df.head()

,year_ceremony,ceremony,category,canon_category,name,film,winner,release_year
8183,2002,74,ACTOR IN A LEADING ROLE,ACTOR IN A LEADING ROLE,Russell Crowe,a beautiful mind,False,2001
8184,2002,74,ACTOR IN A LEADING ROLE,ACTOR IN A LEADING ROLE,Sean Penn,i am sam,False,2001
8185,2002,74,ACTOR IN A LEADING ROLE,ACTOR IN A LEADING ROLE,Will Smith,ali,False,2001
8186,2002,74,ACTOR IN A LEADING ROLE,ACTOR IN A LEADING ROLE,Denzel Washington,training day,True,2001
8187,2002,74,ACTOR IN A LEADING ROLE,ACTOR IN A LEADING ROLE,Tom Wilkinson,in the bedroom,False,2001


## Step 4: Separate Genre, Directors, Cast, Production Companies Rows

This is useful for any analysis based on these variables.


In [ ]:
movie_tidy_genre = movie_df.copy()
movie_tidy_genre["genre"] = movie_tidy_genre["genres"].str.split(", ")
movie_tidy_genre = movie_tidy_genre.explode("genre")
movie_tidy_genre = movie_tidy_genre.drop(columns=['genres'])

In [ ]:
movie_tidy_director = movie_df.copy()
movie_tidy_director["director"] = movie_tidy_director["directors"].str.split(", ")
movie_tidy_director = movie_tidy_director.explode("director")
movie_tidy_director = movie_tidy_director.drop(columns=['directors'])

In [ ]:
movie_tidy_cast = movie_df.copy()
movie_tidy_cast["ind_cast"] = movie_tidy_cast["cast"].str.split(", ")
movie_tidy_cast = movie_tidy_cast.explode("ind_cast")
movie_tidy_cast = movie_tidy_cast.drop(columns=['cast'])

In [ ]:
movie_tidy_prod = movie_df.copy()
movie_tidy_prod["production_company"] = movie_tidy_prod["production_companies"].str.split(", ")
movie_tidy_prod = movie_tidy_prod.explode("production_company")
movie_tidy_prod = movie_tidy_prod.drop(columns=['production_companies'])

In [ ]:
movie_tidy_cast.head(50)

,id,title,vote_average,vote_count,status,revenue,runtime,adult,budget,tconst,...,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,release_year,ind_cast
0,27205,inception,8.364,34495,Released,825532764,148,False,160000000,tt1375666,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2793340,2010,Leonardo DiCaprio
0,27205,inception,8.364,34495,Released,825532764,148,False,160000000,tt1375666,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2793340,2010,Joseph Gordon-Levitt
0,27205,inception,8.364,34495,Released,825532764,148,False,160000000,tt1375666,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2793340,2010,Ken Watanabe
0,27205,inception,8.364,34495,Released,825532764,148,False,160000000,tt1375666,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2793340,2010,Tom Hardy
0,27205,inception,8.364,34495,Released,825532764,148,False,160000000,tt1375666,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2793340,2010,Elliot Page
0,27205,inception,8.364,34495,Released,825532764,148,False,160000000,tt1375666,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2793340,2010,Dileep Rao
0,27205,inception,8.364,34495,Released,825532764,148,False,160000000,tt1375666,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2793340,2010,Cillian Murphy
0,27205,inception,8.364,34495,Released,825532764,148,False,160000000,tt1375666,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2793340,2010,Tom Berenger
0,27205,inception,8.364,34495,Released,825532764,148,False,160000000,tt1375666,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2793340,2010,Marion Cotillard
0,27205,inception,8.364,34495,Released,825532764,148,False,160000000,tt1375666,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2793340,2010,Pete Postlethwaite


# Initiate SQLite

In [ ]:
import sqlite3
conn = sqlite3.connect('/content/movies.db')
movie_df.to_sql('movies', conn, if_exists='replace', index=False)
movie_tidy_cast.to_sql('movie_cast', conn, if_exists='replace', index=False)
movie_tidy_genre.to_sql('movie_genre', conn, if_exists='replace', index=False)
movie_tidy_director.to_sql('movie_director', conn, if_exists='replace', index=False)
movie_tidy_prod.to_sql('movie_prod', conn, if_exists='replace', index=False)
oscar_df.to_sql('oscars', conn, if_exists='replace', index=False)

2927

In [ ]:
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master;")
print(cursor.fetchall())

[('movies',), ('movie_cast',), ('movie_genre',), ('movie_director',), ('movie_prod',), ('oscars',)]


# SQL Analysis


---


# Do production companies earn higher revenue after their first Oscar win?
## SQL Query 1: Groupby, Inner Join, Subquery

**Purpose**

This query compares each production company’s average movie revenue before and after the year of its first Oscar win. This helps evaluate whether an initial Oscar win is associated with a shift in the commercial performance of that company’s films over time. This is a useful first query to have as it **legitimizes the remainder of this analysis:** it is worth it for a producer to think about Oscar awards as there is indeed a correlation between winning one and having more commercial success.

**Methodology/Annotations**

A subquery finds each production company’s first_win_year by joining movie_prod to oscars, filtering to winning films, and taking the earliest release year for each company. That benchmark is then joined back to all films by the same company and the average revenue before and after the first win year is calculated.

**SELECT fw.production_company:** final result should have one row per production company.

**fw:** the alias for the subquery below, an acronym of “first win.”

**First AVG block:** CASE checks each movie row, and if the movie's release_year is earlier than the company's first_win_year, it returns that movie's revenue or NULL oithers. The AVG() function then averages all of them to give the average revenue of movies released before the first Oscar win.

**Second AVG block:** CASE checks each movie row, and if the movie's release_year is later than the company's first_win_year, it returns that movie's revenue or NULL oithers. The AVG() function then averages all of them to give the average revenue of movies released after the first Oscar win.

**FROM:** Starts a subquery

**SELECT..FROM block:** Selects the company name and the earliest release year among its Oscar-winning films via the MIN() function from the movie_prod dataframe. mp is the alias given to movie_prod.

**Subquery INNER JOIN block:** Matches the movie title to the Oscar film title. An inner join was used because we only care about movies that are in the oscars databse. o is the alias given to oscars.

**WHERE block:**: Filters joined rows so that only Oscar-winning films (not just nominated) are included and production company can't be null

**GROUP BY block:** This groups the rows by production company

**Main INNER JOIN block:** Takes the temporary fw table and joins it to every movie row for that company

**GROUP BY:** Groups all the rows that have the same production company

**HAVING block:** Filters to keep companies that have one movie before the first win year and one movie afterwards, so that the outputs calculated won't be null

**ORDER BY:** Sorts the results from highest to lowest average revenue after win

In [ ]:
query_prod_before_after = """
SELECT
    fw.production_company,
    AVG(
        CASE
            WHEN mp.release_year < fw.first_win_year
            THEN mp.revenue
        END
    ) AS avg_revenue_before_win,
    AVG(
        CASE
            WHEN mp.release_year > fw.first_win_year
            THEN mp.revenue
        END
    ) AS avg_revenue_after_win
FROM (
    SELECT
        mp.production_company,
        MIN(mp.release_year) AS first_win_year
    FROM movie_prod mp
    INNER JOIN oscars o
        ON mp.title = o.film
    WHERE o.winner = 1
    GROUP BY mp.production_company
) fw
INNER JOIN movie_prod mp
    ON mp.production_company = fw.production_company
GROUP BY fw.production_company
ORDER BY avg_revenue_after_win DESC;
"""
prod_before_after = pd.read_sql_query(query_prod_before_after, conn) # uses the Pandas real_sql method to process the query
prod_before_after.head(30) # output is exhibited in a dataframe called prod_before_after

,production_company,avg_revenue_before_win,avg_revenue_after_win
0,Fairview Entertainment,2.815745e+08,1.663075e+09
1,Joint Effort,8.623452e+07,1.074458e+09
2,Lightstorm Entertainment,NaN,1.012340e+09
3,Marvel Studios,7.420200e+08,1.008047e+09
4,Second Mate Productions,NaN,9.610000e+08
5,Pascal Pictures,3.370920e+08,8.920963e+08
6,LuckyChap Entertainment,NaN,8.169589e+08
7,Illumination,6.387933e+08,7.327723e+08
8,WingNut Films,8.912744e+06,6.851807e+08
9,Syncopy,1.613145e+08,6.754477e+08


In [ ]:
print("Average revenue before win:", prod_before_after.avg_revenue_before_win.mean())
print("Average revenue after win:", prod_before_after.avg_revenue_after_win.mean())
print(f"Post win revenue premium: {prod_before_after.avg_revenue_after_win.mean() / prod_before_after.avg_revenue_before_win.mean()}x")

Average revenue before win: 97998663.98161025
Average revenue after win: 126106884.67368905
Post win revenue premium: 1.286822488695901x


# Which films got the most nominations?
## SQL Query 2: Groupby

**Purpose**

This query identifies which films received the most Oscar nominations and how many of those nominations resulted in wins. This helps highlight the most recognized films in the Oscars dataset as we begin our analysis.

**Methodology/Annotations**

The query groups the oscars table by film title so that each film is summarized into a single row. It then uses COUNT(*) to calculate total nominations and SUM(o.winner) to calculate total wins, ordering the results first by nomination count and then by win count in descending order.

**COUNT(*)** Counts how many rows each film has in the oscars table

**SUM(o.winner):** Adds up the values in the winner column for each film

**GROUP BY...:** Put together all rows that have the same film title so that rather than having one row per nomination, we have one row per film.


**ORDER BY...:** Sorts the final results via two ranking logics, the first by nomination count from highest to lowest, then win count from highest to lowest.

In [ ]:
ranked_noms_query = """
SELECT
    o.film,
    COUNT(*) AS nomination_count,
    SUM(o.winner) AS win_count
FROM oscars o
GROUP BY o.film
ORDER BY nomination_count DESC, win_count DESC;
"""
ranked_nominations_df = pd.read_sql_query(ranked_noms_query, conn)
ranked_nominations_df.head(50)

,film,nomination_count,win_count
0,None,185,185
1,titanic,14,11
2,la la land,14,6
3,oppenheimer,13,7
4,shakespeare in love,13,7
5,chicago,13,6
6,forrest gump,13,6
7,the lord of the rings the fellowship of the ring,13,4
8,the shape of water,13,4
9,the curious case of benjamin button,13,3


# Which genre has the most nominations?
## SQL Query 3: Inner Join & Groupby

**Purpose**

This query identifies which genres are associated with the most Oscar nominations by counting nomination records linked to films in each genre. This helps show which genres appear most often in the Oscars dataset overall.

**Methodology/Annotations**

An INNER JOIN matches movie_genre to oscars by film title and then  rows are grouped by genre and uses COUNT(*) to total the number of nomination records associated with each genre.

**SELECT...COUNT(*):** Counts how many joined rows there are for each genre. total_nominations is the alias for the result column

**INNER JOIN block**: Joins the movie_genre and oscars tables when a film is in both of them by matching the title of the film.

**WHERE mg.genre IS NOT NULL:** Gets rid of any rows with missing genres

**GROUP BY mg.genre:** Combines many movie-level rows into one summary row per genre.

**ORDER BY...:** Sorts the final result by total nominations from highest to lowest.





In [ ]:
query_genre_nominations_total = """
SELECT
    mg.genre,
    COUNT(*) AS total_nominations
FROM movie_genre mg
JOIN oscars o
    ON mg.title = o.film
GROUP BY mg.genre
ORDER BY total_nominations DESC;
"""
genre_nominations_total_df = pd.read_sql_query(query_genre_nominations_total, conn)
genre_nominations_total_df.head(20)

,genre,total_nominations
0,Drama,2596
1,Romance,798
2,Comedy,721
3,Adventure,691
4,Thriller,586
5,Action,586
6,History,568
7,Crime,426
8,Fantasy,390
9,Science Fiction,336


# Which categories have the highest nomination rate?
## SQL Query 4: Left Join & Groupby

**Purpose**

This query calculates the Oscar nomination rate for each genre by measuring the share of movies in that genre that received at least one nomination. This is a helpful insight because it normalizes nominations by the total number of films in each genre, making it possible to compare genres fairly rather than just counting which genres have the most nominations overall. The result helps identify which genres are most likely to produce Oscar-nominated films relative to their size in the full movie dataset.

**Methodology/Annotations**

A LEFT JOIN matches movie_genre to oscars by title while keeping all genre rows, so the denominator includes all movies in each genre. The query then computes the proportion of unique movies in each genre that had at least one Oscar match.

**SELECT block:** The goal is to calculate the ratio of movies in each genre which have received at least one Oscar nomination. The output column is given the alias nomination_rate

**COUNT(DISTINCT CASE...):** Counts how many unique movie titles in that genre had at least one Oscar match. Distinct makes it so that movies with multiple Oscar nominations aren't double-counted.

**CASE WHEN o.film IS NOT NULL THEN mg.title END:** Checks whether a movie from movie_genre has a match in oscars and returns the movie title if so, else null.

***1.0:** Forces decimal division (ChatGPT suggested this while I was debugging)

**/ COUNT(mg.title):** Counts how many rows there are in movie_genre for that genre

**LEFT JOIN oscars o**: Used a left join here because we want to keep all rows from movie_genre, even if there isn't a matchg in oscars because we need it for the denominator.

**ON mg.title = o.film:** Join condition which matches titles

**GROUP BY mg.genre:** Groups all rows by genre so that the final output has one row per genre

**ORDER BY nomination_rate DESC;:** Sorts the results from highest nomination rate to lowest.

In [ ]:
nom_rate_query = """
SELECT
    mg.genre,
    COUNT(DISTINCT CASE WHEN o.film IS NOT NULL THEN mg.title END) * 1.0/ COUNT(mg.title) AS nomination_rate
FROM movie_genre mg
LEFT JOIN oscars o
    ON mg.title = o.film
GROUP BY mg.genre
ORDER BY nomination_rate DESC;
"""
avg_nominations_per_genre_df = pd.read_sql_query(nom_rate_query, conn)
avg_nominations_per_genre_df.head(20)

,genre,nomination_rate
0,Animation,0.181967
1,Music,0.152120
2,Fantasy,0.142857
3,History,0.139111
4,Adventure,0.137265
5,Western,0.136986
6,Family,0.135135
7,Drama,0.128852
8,War,0.128631
9,Documentary,0.123457


# Do nominated films generate more revenue than the wider pool?


## SQL Query 5: Inner Join

**Purpose**

This query creates a joined dataset of films that appear in both the movies table and the oscars table, meaning films with at least one Oscar nomination record. This is a helpful starting point for investigating whether nominated films generate more revenue than the wider pool because it combines revenue from the movies database with Oscar-level variables like category, nominee name, and winner status in one table. The resulting dataset is then used to compare the commercial performance of nominated films against the broader movie dataset in the Python EDA Analysis.

**Methology/Annotations**

An inner join was used because it only keeps rows where both tables have a match based on the join condition, which matches with the objective of the query, which is to pull information from the movie database for all movies in the oscar database.

**SELECT block:** Identified many columns which are desired to be in the joined datraframe.

**INNER JOIN:** An inner join was used because we only want to look at stats for films that have at least one nomination.

In [ ]:
query = """
SELECT
    m.title,
    m.release_date,
    m.vote_average,
    m.revenue,
    o.category,
    o.canon_category,
    o.name,
    o.film,
    o.winner
FROM movies m
INNER JOIN oscars o
  ON m.title = o.film
"""
nominated_movies_df = pd.read_sql_query(query, conn)
nominated_movies_df.head()

,title,release_date,vote_average,revenue,category,canon_category,name,film,winner
0,inception,2010-07-15 00:00:00,8.364,825532764,ART DIRECTION,ART DIRECTION,Guy Hendrix Dyas/Larry Dias/Doug Mowat,inception,0
1,inception,2010-07-15 00:00:00,8.364,825532764,BEST PICTURE,BEST PICTURE,Emma Thomas/Christopher Nolan,inception,0
2,inception,2010-07-15 00:00:00,8.364,825532764,CINEMATOGRAPHY,CINEMATOGRAPHY,Wally Pfister,inception,1
3,inception,2010-07-15 00:00:00,8.364,825532764,MUSIC (Original Score),MUSIC (Original Score),Hans Zimmer,inception,0
4,inception,2010-07-15 00:00:00,8.364,825532764,SOUND EDITING,SOUND EDITING,Richard King,inception,1


# How is revenue for nominated films distributed across quartiles formed from nominated films?

## SQL Query 6: Window Function

**Purpose**

This query determines the percent of Oscar-nominated films that fall into each revenue quartile when quartiles are based on all films within the same genre. This is a helpful insight because it measures whether nominated films tend to cluster among the highest earners in their genre rather than being spread evenly across the revenue distribution.

**Methodology/Annotations**
A window function was used to compute within-genre revenue quartiles and add the result to a new column, revenue_quartile, which is then used to summarize where nominated films fall. The result is  interesting because it directly addresses whether Oscar recognition is concentrated among commercially successful films.

**WITH all_films_ranked AS (...):** Starts a common table expression that builds revenue quartiles using all films in each genre, not just nominated films.

**SELECT DISTINCT:** Selects unique movie-genre pairs so the same movie is not repeated within the same genre before quartiles are assigned

**NTILE(4):** Divides films into 4 groups as evenly as possible

**OVER (PARTITION BY mg.genre ORDER BY mg.revenue DESC):** Tells SQL to create the 4 groups separately within each genre, ordering films from highest revenue to lowest

**WHERE mg.genre IS NOT NULL AND mg.revenue IS NOT NULL:** Removes rows missing genre or revenue so quartiles can be assigned correctly

**WITH nominated_films AS (...):** Starts a second common table expression containing only distinct Oscar-nominated films

**SELECT DISTINCT film:** Keeps each nominated film only once, even if it received multiple nominations

**FROM oscars:** Pulls nominated films from the Oscars table

**WHERE film IS NOT NULL:** Excludes Oscar rows with missing film titles.

**1st SELECT**: Summarizes where nominated films fall across the revenue quartiles built from all films

**afr.revenue_quartile:** Returns the quartile number from the ranked all-films table

**COUNT(*) AS nominated_films:** Counts how many nominated films fall into each revenue quartile.

**ROUND(COUNT() * 100.0 / SUM(COUNT()) OVER (), 2):** Calculates the percent of nominated films in each quartile

**COUNT(*) * 100.0** converts the quartile count into a percentage numerator

**SUM(COUNT(*)) OVER ()** adds the quartile counts across all quartiles

**ROUND(..., 2)** rounds the percentage to 2 decimal places

**FROM all_films_ranked afr:** Uses the table where all films have already been assigned a within-genre revenue quartile

**GROUP BY afr.revenue_quartile:** Groups the matched rows by quartile so each quartile gets one summary row

**ORDER BY afr.revenue_quartile:** Sorts the output from quartile 1 through quartile 4

In [ ]:
query = """
WITH all_films_ranked AS (
    SELECT DISTINCT
        mg.title,
        mg.genre,
        mg.revenue,
        NTILE(4) OVER (
            PARTITION BY mg.genre
            ORDER BY mg.revenue DESC
        ) AS revenue_quartile
    FROM movie_genre mg
),
nominated_films AS (
    SELECT DISTINCT
        film
    FROM oscars
)
SELECT
    afr.revenue_quartile,
    COUNT(*) AS nominated_films,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_of_nominated_films
FROM all_films_ranked afr
JOIN nominated_films nf
    ON afr.title = nf.film
GROUP BY afr.revenue_quartile
ORDER BY afr.revenue_quartile;
"""
revenue_quartile_df = pd.read_sql_query(query, conn)
revenue_quartile_df.head()

,revenue_quartile,nominated_films,pct_of_nominated_films
0,1,1790,57.35
1,2,812,26.02
2,3,384,12.30
3,4,135,4.33


# Did the winning film in a category have the highest revenue compared to all nominees in that category?

## SQL Query 7: Window Function

**Purpose**

This query determines the percent of winning films that have the highest revenue compared to other nomations in its category.This is a helpful insight because it measures whether commercial success and award success line up inside each category.

**Methodology**

A window function was used to compute a rank across relatewd rows and add the result to a new column, revenue_rank_in_category, which is needed for the later percentage calculation.

**Annotations**

**Outer SELECT:** Computes one final percentage from the subquery result.

**SUM(CASE WHEN sub.winner = 1 AND sub.revenue_rank_in_category = 1 THEN 1 ELSE 0 END):** Counts how many rows are Oscar winners and have revenue rank 1 within their category. Since rank 1 means highest revenue, this is counting winning films that were the top earners in their category

**CAST(... AS REAL) * 100:** Converts the count into decimal math and multiplies by 100 so the final answer is a percentage.

**/ SUM(CASE WHEN sub.winner = 1 THEN 1 ELSE 0 END):** Divides by the total number of winning-film rows in the subquery to give the share of winners that were also the highest-revenue film in their category.

**FROM ( ... ) AS sub:** Starts a subquery and labels it sub

**Subquery SELECT:** Builds one row per movie-category Oscar record and assigns a revenue rank within category

**ROW_NUMBER() OVER (...):** Window function that assigns a unique rank to each row within a category based on revenue.

**PARTITION BY o.canon_category:** Tells SQL to restart the ranking separately for each Oscar category. So Best Picture films are ranked against other Best Picture films, Best Director films against other Best Director films, etc.

**ORDER BY m.revenue DESC:** Within each category, films are ordered from highest revenue to lowest

**INNER JOIN oscars o:** Joins movie records to Oscar records. An inner join was used because the query only cares about films that appear in both the movie dataset and the Oscars dataset. Unmatched films are irrelevant to this question.

In [ ]:
query_winning_revenue_rank = """
SELECT
    CAST(SUM(CASE WHEN sub.winner = 1 AND sub.revenue_rank_in_category = 1 THEN 1 ELSE 0 END) AS REAL) * 100
    / SUM(CASE WHEN sub.winner = 1 THEN 1 ELSE 0 END) AS percentage_winning_films_with_highest_revenue
FROM (
    SELECT
        m.title,
        o.canon_category,
        o.winner,
        m.revenue,
        ROW_NUMBER() OVER (
            PARTITION BY o.canon_category
            ORDER BY m.revenue DESC
        ) AS revenue_rank_in_category
    FROM movies m
    INNER JOIN oscars o
        ON m.title = o.film
) AS sub;
"""

percentage_sql_no_cte_df = pd.read_sql_query(query_winning_revenue_rank, conn)
display(percentage_sql_no_cte_df)

,percentage_winning_films_with_highest_revenue
0,1.631702


# Do movies by the same director make more money after the director has won their first Oscar?

## SQL Query 8: Join Directors to Oscar-Winning Films

**Purpose**

This query links directors from the movie dataset with Oscar-winning films in the Oscar dataset.

**Methodology/Annotations**

An *Inner Join* is used to match movie titles between the movies table and the oscars table. It is used because the goal is to analyze only movies that appear in both datasets. The join condition compares m.title and o.film. Only rows where the movie exists in both tables are kept. The query then filters the results to include only rows where the Oscar record indicates a winning nomination.

In [ ]:
query_director_join = """
SELECT DISTINCT
    md.director,
    md.title,
    md.release_year,
    md.revenue,
    o.category,
    o.winner
FROM movie_director md
JOIN oscars o
    ON md.title = o.film
WHERE o.winner = 1
ORDER BY md.director;
"""

director_winners = pd.read_sql_query(query_director_join, conn)
director_winners.head(50)

,director,title,release_year,revenue,category,winner
0,Adam McKay,the big short,2015,133346506,WRITING (Adapted Screenplay),1
1,Adam McKay,vice,2018,76073488,MAKEUP AND HAIRSTYLING,1
2,Adrian Molina,coco,2017,800526015,ANIMATED FEATURE FILM,1
3,Adrian Molina,coco,2017,800526015,MUSIC (Original Song),1
4,Alejandro Amenábar,the sea inside,2004,38535221,FOREIGN LANGUAGE FILM,1
5,Alejandro G. Iñárritu,babel,2006,135330182,MUSIC (Original Score),1
6,Alejandro G. Iñárritu,birdman or the unexpected virtue of ignorance,2014,103215094,CINEMATOGRAPHY,1
7,Alejandro G. Iñárritu,birdman or the unexpected virtue of ignorance,2014,103215094,DIRECTING,1
8,Alejandro G. Iñárritu,birdman or the unexpected virtue of ignorance,2014,103215094,BEST PICTURE,1
9,Alejandro G. Iñárritu,birdman or the unexpected virtue of ignorance,2014,103215094,WRITING (Original Screenplay),1


This query identifies directors whose films have won Oscars by linking the movie dataset with the Oscar nomination dataset. The resulting dataset provides information about the directors, the films they directed, the revenue of those films, and the categories in which they won.

## SQL Query 9: Subquery

**Purpose**

This query compares each director’s average movie revenue before and after the year of their first Oscar win. This is a helpful insight because it measures whether winning an Oscar is associated with a change in the commercial performance of a director’s films over time.

**Methodology/Annotations**

A subquery is used to identify each director’s first_win_year by finding the earliest release year among that director’s Oscar-winning films. That first-win benchmark is then joined back to all films by the same director so the query can separate movies released before the first win from movies released after it.

**SELECT fw.director:** Final result should have one row per director.

**fw:** Alias for the subquery below, short for “first win.”

**First AVG block**: CASE checks each movie row, and if the movie’s release_year is earlier than the director’s first_win_year, it returns that movie’s revenue; otherwise it returns NULL. AVG() then averages those revenues to get the director’s average revenue before the first Oscar win.

**Second AVG block:** CASE checks each movie row, and if the movie’s release_year is later than the director’s first_win_year, it returns that movie’s revenue; otherwise it returns NULL. AVG() then averages those revenues to get the director’s average revenue after the first Oscar win.

**FROM:** Starts a subquery.

**SELECT...FROM block:** Selects the director name and the earliest release year among that director’s Oscar-winning films using MIN(md.release_year) from movie_director. md is the alias for movie_director.

**Subquery INNER JOIN block:** Matches movie titles in movie_director to film titles in oscars. An inner join is used because we only care about movies that appear in the Oscars dataset. o is the alias for oscars.

**WHERE block:** Filters joined rows so only Oscar-winning films are included (o.winner = 1) and director cannot be null.

**GROUP BY block**: Groups rows by director so each director gets one first_win_year.

**Main JOIN block:** Joins the temporary fw table back to all rows in movie_director for that same director.

**Second WHERE block:** Filters out rows where director or release year is null.

**Main GROUP BY:** Groups all rows by director so the before/after revenue averages are calculated per director.

**HAVING block:** Keeps only directors who have at least one movie before the first win year and at least one movie after it, so both averages are meaningful.

**ORDER BY:** Sorts results from highest to lowest average revenue after win.

In [ ]:
query_director_before_after = """
SELECT
    fw.director,
    AVG(
        CASE
            WHEN md.release_year < fw.first_win_year
            THEN md.revenue
        END
    ) AS avg_revenue_before_win,
    AVG(
        CASE
            WHEN md.release_year > fw.first_win_year
            THEN md.revenue
        END
    ) AS avg_revenue_after_win
FROM (
    SELECT
        md.director,
        MIN(md.release_year) AS first_win_year
    FROM movie_director md
    JOIN oscars o
        ON md.title = o.film
    WHERE o.winner = 1
    GROUP BY md.director
) fw
JOIN movie_director md
    ON md.director = fw.director
GROUP BY fw.director
HAVING avg_revenue_before_win IS NOT NULL
   AND avg_revenue_after_win IS NOT NULL
ORDER BY avg_revenue_after_win DESC;
"""

director_before_after = pd.read_sql_query(query_director_before_after, conn)
director_before_after

,director,avg_revenue_before_win,avg_revenue_after_win
0,Jon Favreau,2.465072e+08,1.663075e+09
1,Greta Gerwig,7.896649e+07,1.428545e+09
2,James Cameron,7.837120e+07,1.216292e+09
3,J.J. Abrams,3.984795e+08,9.674573e+08
4,Lee Unkrich,4.973669e+08,9.359438e+08
...,...,...,...
125,Paolo Sorrentino,1.152567e+07,2.000000e+06
126,Alfonso Cuarón,1.607306e+08,1.140769e+06
127,Paul Schrader,5.027580e+05,1.097308e+06
128,David Ayer,7.090291e+07,9.426660e+05


In [ ]:
print("Average revenue before win:", director_before_after.avg_revenue_before_win.mean())
print("Average revenue after win:", director_before_after.avg_revenue_after_win.mean())
print(f"Post win revenue premium: {director_before_after.avg_revenue_after_win.mean() / director_before_after.avg_revenue_before_win.mean()}x")

Average revenue before win: 133644308.8269486
Average revenue after win: 294688126.9981758
Post win revenue premium: 2.2050181529223014x


# Do movies by the same actor/actress make more money after the actor/actress has won their first Oscar?

## SQL Query 10: Subquery

**Purpose**

This query compares each actor’s average movie revenue before and after the year of their first Oscar win. This is a helpful insight because it measures whether winning an Oscar is associated with a change in the commercial performance of an actor’s films over time.

**Methodology/Annotations**

A subquery is used to identify each actor’s first_win_year by finding the earliest release year among that actor’s Oscar-winning records in the oscars table. That first-win benchmark is then joined back to all films by the same actor in movie_cast so the query can separate movies released before the first win from movies released after it.

**SELECT fw.actor:** Final result should have one row per actor.

**fw:** Alias for the subquery below, short for “first win.”

**First AVG block:** CASE checks each movie row, and if the movie’s release_year is earlier than the actor’s first_win_year, it returns that movie’s revenue; otherwise it returns NULL. AVG() then averages those revenues to get the actor’s average revenue before the first Oscar win.

**Second AVG block**: CASE checks each movie row, and if the movie’s release_year is later than the actor’s first_win_year, it returns that movie’s revenue; otherwise it returns NULL. AVG() then averages those revenues to get the actor’s average revenue after the first Oscar win.

**FROM**: Starts a subquery.

**SELECT...FROM block:** Selects the actor name and the earliest release year among that actor’s Oscar-winning records using MIN(release_year) from oscars. name is renamed as actor.

**Subquery WHERE block:** Filters rows so only Oscar-winning records are included (winner = 1) and actor name cannot be null.

**GROUP BY block:** Groups rows by actor so each actor gets one first_win_year.

**Main JOIN block:** Joins the temporary fw table to all rows in movie_cast for that same actor using mc.ind_cast = fw.actor.

**Second WHERE block: **Filters out rows where actor name or release year is null.

**Main GROUP BY:** Groups all rows by actor so the before/after revenue averages are calculated per actor.

**HAVING block**: Keeps only actors who have at least one movie before the first win year and at least one movie after it, so both averages are meaningful.

**ORDER BY:** Sorts results from highest to lowest average revenue after win.

In [ ]:
query_actor_before_after = """
SELECT
    fw.actor,
    AVG(
        CASE
            WHEN mc.release_year < fw.first_win_year
            THEN mc.revenue
        END
    ) AS avg_revenue_before_win,
    AVG(
        CASE
            WHEN mc.release_year > fw.first_win_year
            THEN mc.revenue
        END
    ) AS avg_revenue_after_win
FROM (
    SELECT
        name AS actor,
        MIN(release_year) AS first_win_year
    FROM oscars
    WHERE winner = 1
    GROUP BY name
) fw
JOIN movie_cast mc
    ON mc.ind_cast = fw.actor
GROUP BY fw.actor
HAVING avg_revenue_before_win IS NOT NULL
   AND avg_revenue_after_win IS NOT NULL
ORDER BY avg_revenue_after_win DESC;
"""

query_actor_before_after = pd.read_sql_query(query_actor_before_after, conn)
query_actor_before_after

,actor,avg_revenue_before_win,avg_revenue_after_win
0,Jordan Peele,7.571640e+07,1.073395e+09
1,Laura Dern,9.659727e+07,1.001978e+09
2,James Earl Jones,1.324339e+08,8.322177e+08
3,Angela Lansbury,1.634357e+08,5.085753e+08
4,Rami Malek,1.490964e+08,4.973016e+08
...,...,...,...
133,Samuel L. Jackson,2.132066e+08,2.148819e+07
134,Gary Oldman,2.398673e+08,1.589746e+07
135,Joaquin Phoenix,1.026545e+08,7.150000e+06
136,Jack Palance,1.731614e+08,4.885850e+06


In [ ]:
print("Average revenue before win:", query_actor_before_after.avg_revenue_before_win.mean())
print("Average revenue after win:", query_actor_before_after.avg_revenue_after_win.mean())
print(f"Post win revenue premium: {query_actor_before_after.avg_revenue_after_win.mean() / query_actor_before_after.avg_revenue_before_win.mean()}x")

Average revenue before win: 132136922.44202669
Average revenue after win: 208595876.17195588
Post win revenue premium: 1.5786342856855513x
